In [191]:
import pandas as pd     # Ипортирование библиотеки

# Загрузка файлов
main_data = pd.read_json("./src/competitors2.json").T.reset_index() # Создание таблицы из 1 файла
result_data = pd.read_table("./src/results_RUN.txt", sep=' ', header=None) # Создание таблицы из 2 файла

In [192]:
# Переименование колонок
main_data.columns = ['Нагрудный номер', 'Имя', 'Фамилия']
result_data.columns = ['Нагрудный номер', 'Статус', 'Время']

In [193]:
result_data['Время'] = result_data['Время'].str.split(',').str.get(0) # Разделение времени по разделютелю запятой, 1 значение заносится в переменную
result_data = result_data.reset_index(drop='index') # Обновление индексов

In [194]:
from datetime import datetime # Импортирование библиотеки

start_index = result_data.loc[result_data['Статус']=='start'].index # Индексы строчек где хранятся времена старта
finish_index = result_data.loc[result_data['Статус']=='finish'].index # Индексы строчек где хранятся времена финиша 
reslist = [] # Пустой список, куда будем записывать результаты

for i in range(0, len(start_index)): 
    start = datetime.strptime(result_data['Время'].loc[start_index[i]], '%H:%M:%S') # Занос в переменную время старта
    end =  datetime.strptime(result_data['Время'].loc[finish_index[i]], '%H:%M:%S') # Занос в переменную время финиша
    result = end - start # Просчет среднего времени
    reslist.append(result) # Добавление в результов списка

temp = pd.DataFrame(list(reslist), columns=['Итоговое время']) # Создание таблицы "Итоговое время"

In [195]:
result_data.drop(result_data.loc[result_data['Статус']=='start'].index, inplace=True)   # Удаление всех строчке содержащих start
result_data = result_data.reset_index(drop='index') # Обновление индексов

In [196]:
result_data = result_data.join(temp) # Добавление итогового времени в таблицу с результатами

In [197]:
del result_data['Статус']   # Удаление столбца Статус
del result_data['Время']    # Удаление столбца Время

In [198]:
finish_data = main_data.merge(result_data, on='Нагрудный номер', how='left') # Соединение таблиц, по ключу "Нагрудный номер"

In [199]:
finish_data['Итоговое время'] = finish_data['Итоговое время'].astype('string') # Перевод столбца в тип данных string
finish_data['Итоговое время'] = finish_data['Итоговое время'].str.split(' ').str.get(2) # Вычленение из столбца только время

In [200]:
finish_data = finish_data.sort_values(by='Итоговое время', ascending=True) # Сортировка по Убыванию в столбце "Итоговое время"
finish_data = finish_data.reset_index(drop='index') # Обновление индексов

In [202]:
print(finish_data) # Вывод финальной таблицы

     Нагрудный номер         Имя    Фамилия Итоговое время
0                122     Шарапов      Игнат       00:01:08
1                 59     Явлюхин      Роман       00:01:08
2                 34      Зверев     Адриан       00:01:09
3                160      Щукина     Регина       00:01:09
4                103       Быков  Станислав       00:01:10
..               ...         ...        ...            ...
295              159     Бородай       Дина       00:08:05
296              196      Карпов     Гордей       00:08:12
297               62  Андрусейко      Ольга       00:08:12
298               36       Шуста      Артём       00:08:17
299              266    Агафонов   Афанасий           <NA>

[300 rows x 4 columns]
